In [35]:
import numpy as np
import pandas as pd
from scipy import optimize
import matplotlib.pyplot as pltI

In [36]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def sigmoidGradient(z):
    s = sigmoid(z)
    return s * (1 - s)

def randInitializeWeights(L_in, L_out, epsilon_init=None):
    if epsilon_init is None:
        epsilon_init = np.sqrt(6) / np.sqrt(L_in + L_out)
    return np.random.rand(L_out, 1 + L_in) * 2 * epsilon_init - epsilon_init

def flatten_params(Theta1, Theta2):
    return np.concatenate([Theta1.ravel(), Theta2.ravel()])

def reshape_params(nn_params, input_layer_size, hidden_layer_size, num_labels):
    t1_end = hidden_layer_size * (input_layer_size + 1)
    Theta1 = nn_params[:t1_end].reshape(hidden_layer_size, input_layer_size + 1)
    Theta2 = nn_params[t1_end:].reshape(num_labels, hidden_layer_size + 1)
    return Theta1, Theta2

In [37]:
def cargar_glass_csv(glass):
    df = pd.read_csv("data/glass.csv")

    # Detectar columna objetivo (común: 'Type')
    posibles_y = [c for c in df.columns if c.lower() in ('type','class','target','y')]
    y_col = posibles_y[0] if posibles_y else df.columns[-1]

    # Quitar Id si existe
    drop_cols = [c for c in df.columns if c.lower() in ('id','index')]
    X_df = df.drop(columns=drop_cols + [y_col], errors='ignore')
    y = df[y_col].values

    # Mapear clases originales -> 0..K-1
    clases = np.unique(y)
    clases_ordenadas = np.sort(clases)
    label2id = {c:i for i,c in enumerate(clases_ordenadas)}
    id2label = {i:c for c,i in label2id.items()}
    y_id = np.vectorize(label2id.get)(y)

    X = X_df.values.astype(np.float64)
    return X, y_id, label2id, id2label

def train_val_split(X, y, test_size=0.2, seed=42):
    np.random.seed(seed)
    m = X.shape[0]
    idx = np.random.permutation(m)
    cut = int((1 - test_size) * m)
    tr, va = idx[:cut], idx[cut:]
    return X[tr], y[tr], X[va], y[va]

def estandarizar_train_val(Xtr, Xva):
    mu = Xtr.mean(axis=0)
    sigma = Xtr.std(axis=0, ddof=0)
    sigma[sigma == 0] = 1.0
    return (Xtr - mu) / sigma, (Xva - mu) / sigma, (mu, sigma)


In [38]:
def nnCostFunction(nn_params, input_layer_size, hidden_layer_size, num_labels, X, y, lambda_):
    m = X.shape[0]
    Theta1, Theta2 = reshape_params(nn_params, input_layer_size, hidden_layer_size, num_labels)

    # Forward
    a1 = np.concatenate([np.ones((m,1)), X], axis=1)   # (m, n+1)
    z2 = a1.dot(Theta1.T)                              # (m, h)
    a2 = sigmoid(z2)
    a2 = np.concatenate([np.ones((m,1)), a2], axis=1)  # (m, h+1)
    z3 = a2.dot(Theta2.T)                              # (m, K)
    a3 = sigmoid(z3)                                   # (m, K)

    # One-hot
    Y = np.eye(num_labels)[y]                          # (m, K)

    # Costo
    eps = 1e-12
    J = (-1/m) * np.sum(Y*np.log(a3+eps) + (1-Y)*np.log(1-a3+eps))
    reg = (lambda_/(2*m))*(np.sum(Theta1[:,1:]**2) + np.sum(Theta2[:,1:]**2))
    J += reg

    # Backprop
    delta3 = a3 - Y                                    # (m, K)
    delta2 = delta3.dot(Theta2)[:,1:] * sigmoidGradient(z2)  # (m, h)

    Delta1 = delta2.T.dot(a1) / m                      # (h, n+1)
    Delta2 = delta3.T.dot(a2) / m                      # (K, h+1)

    # Regularización (no al sesgo)
    Delta1[:,1:] += (lambda_/m) * Theta1[:,1:]
    Delta2[:,1:] += (lambda_/m) * Theta2[:,1:]

    grad = flatten_params(Delta1, Delta2)
    return J, grad


In [39]:

def predict(Theta1, Theta2, X):
    if X.ndim == 1: X = X[None,:]
    m = X.shape[0]
    a1 = np.concatenate([np.ones((m,1)), X], axis=1)
    z2 = a1.dot(Theta1.T)
    a2 = sigmoid(z2)
    a2 = np.concatenate([np.ones((m,1)), a2], axis=1)
    z3 = a2.dot(Theta2.T)
    a3 = sigmoid(z3)
    return np.argmax(a3, axis=1)

def accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred) * 100.0

def entrenar_red(Xtr, ytr, input_layer_size, hidden_layer_size, num_labels,
                 lambda_=0.1, maxiter=300, seed=0):
    np.random.seed(seed)
    Theta1 = randInitializeWeights(input_layer_size, hidden_layer_size)
    Theta2 = randInitializeWeights(hidden_layer_size, num_labels)
    nn_params0 = flatten_params(Theta1, Theta2)

    def cost_func(p):
        return nnCostFunction(p, input_layer_size, hidden_layer_size, num_labels, Xtr, ytr, lambda_)

    res = optimize.minimize(
        fun=lambda p: cost_func(p)[0],
        x0=nn_params0,
        jac=lambda p: cost_func(p)[1],
        method='L-BFGS-B',
        options={'maxiter': maxiter, 'disp': True}
    )
    Theta1_opt, Theta2_opt = reshape_params(res.x, input_layer_size, hidden_layer_size, num_labels)
    return Theta1_opt, Theta2_opt, res



In [40]:

if __name__ == "__main__":
    X, y, label2id, id2label = cargar_glass_csv("glass.csv")

    Xtr, ytr, Xva, yva = train_val_split(X, y, test_size=0.2, seed=7)
    Xtr_s, Xva_s, stats = estandarizar_train_val(Xtr, Xva)

    input_layer_size  = Xtr_s.shape[1]
    hidden_layer_size = 16      
    num_labels        = len(np.unique(y))
    lambda_ = 0.1

    Theta1, Theta2, info = entrenar_red(
        Xtr_s, ytr,
        input_layer_size, hidden_layer_size, num_labels,
        lambda_=lambda_, maxiter=300, seed=1
    )
    pred_tr = predict(Theta1, Theta2, Xtr_s)
    pred_va = predict(Theta1, Theta2, Xva_s)
    print(f"Accuracy train: {accuracy(ytr, pred_tr):.2f}%")
    print(f"Accuracy val:   {accuracy(yva, pred_va):.2f}%")




Accuracy train: 95.32%
Accuracy val:   67.44%


In [ ]:
# Modificación: entrenamiento con registro de costo y gráfico de resultados
import matplotlib.pyplot as plt
from collections import deque
def entrenar_red_con_hist(Xtr, ytr, input_layer_size, hidden_layer_size, num_labels,
                 lambda_=0.1, maxiter=300, seed=0):
    np.random.seed(seed)
    Theta1 = randInitializeWeights(input_layer_size, hidden_layer_size)
    Theta2 = randInitializeWeights(hidden_layer_size, num_labels)
    nn_params0 = flatten_params(Theta1, Theta2)
    history = []
    def cost_func(p):
        J, grad = nnCostFunction(p, input_layer_size, hidden_layer_size, num_labels, Xtr, ytr, lambda_)
        history.append(J)
        return J, grad
    res = optimize.minimize(
        fun=lambda p: cost_func(p)[0],
        x0=nn_params0,
        jac=lambda p: cost_func(p)[1],
        method='L-BFGS-B',
        options={'maxiter': maxiter, 'disp': True}
    )
    Theta1_opt, Theta2_opt = reshape_params(res.x, input_layer_size, hidden_layer_size, num_labels)
    return Theta1_opt, Theta2_opt, res, history

# Entrenamiento y gráfico
X, y, label2id, id2label = cargar_glass_csv("glass.csv")
Xtr, ytr, Xva, yva = train_val_split(X, y, test_size=0.2, seed=7)
Xtr_s, Xva_s, stats = estandarizar_train_val(Xtr, Xva)
input_layer_size  = Xtr_s.shape[1]
hidden_layer_size = 16
num_labels        = len(np.unique(y))
lambda_ = 0.1
Theta1, Theta2, info, history = entrenar_red_con_hist(
    Xtr_s, ytr, input_layer_size, hidden_layer_size, num_labels, lambda_=lambda_, maxiter=300, seed=1)
pred_tr = predict(Theta1, Theta2, Xtr_s)
pred_va = predict(Theta1, Theta2, Xva_s)
acc_tr = accuracy(ytr, pred_tr)
acc_va = accuracy(yva, pred_va)
plt.figure(figsize=(8,4))
plt.plot(history, label='Costo (loss)')
plt.xlabel('Iteración')
plt.ylabel('Costo')
plt.title('Evolución del costo durante el entrenamiento')
plt.legend()
plt.show()
print(f"Accuracy train: {acc_tr:.2f}%")
print(f"Accuracy val:   {acc_va:.2f}%")